In [1]:
import pandas as pd
df = pd.read_csv('rfm_base.csv', parse_dates=['last_order'])
print(len(df))
df.head()

93358


,customer_unique_id,last_order,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,1,129.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,1,18.90
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,1,69.00
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,1,25.99
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,1,180.00


In [3]:
ref = df['last_order'].max() + pd.Timedelta(days=1)
df['recency'] = (ref - df['last_order']).dt.days

df['R'] = pd.qcut(df['recency'].rank(method='first'), 4, labels=[4,3,2,1]).astype(int)
df['M'] = pd.qcut(df['monetary'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)

def segment(row):
    if row['frequency'] >= 2 and row['R'] >= 3: return 'Loyal'
    if row['frequency'] >= 2: return 'Repeat - At Risk'
    if row['R'] >= 3 and row['M'] >= 3: return 'New High-Value'
    if row['R'] >= 3: return 'New'
    if row['M'] >= 3: return 'Lapsed High-Value'
    return 'Lost'

df['segment'] = df.apply(segment, axis=1)

summary = df.groupby('segment').agg(
    customers=('customer_unique_id', 'count'),
    avg_spend=('monetary', 'mean'),
    total_revenue=('monetary', 'sum')
).round(2).sort_values('total_revenue', ascending=False)
summary['pct_customers'] = (100 * summary['customers'] / summary['customers'].sum()).round(1)
summary['pct_revenue'] = (100 * summary['total_revenue'] / summary['total_revenue'].sum()).round(1)
print(summary)

df.to_csv('rfm_segments.csv', index=False)

                   customers  avg_spend  total_revenue  pct_customers  \
segment                                                                 
New High-Value         22292     231.65     5163841.94           23.9   
Lapsed High-Value      22044     233.69     5151392.18           23.6   
Lost                   23377      47.23     1103998.73           25.0   
New                    22844      47.01     1073856.51           24.5   
Loyal                   1543     264.28      407783.58            1.7   
Repeat - At Risk        1258     254.87      320625.17            1.3   

                   pct_revenue  
segment                         
New High-Value            39.1  
Lapsed High-Value         39.0  
Lost                       8.4  
New                        8.1  
Loyal                      3.1  
Repeat - At Risk           2.4  
